In [1]:
#1. Librerías.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
import re
import unidecode

In [2]:
#2. Constantes.
df_llm_path = "../5-LLMs/openai/pruebas_batch/post_procesamiento_df_final_IL_1610_nuevasvariablesinternacionales_s1500.csv"
df_base_path = "."
df_exportacion_path = "."

In [ ]:
#3. Lectura.
df_llm = pd.read_csv(df_llm_path)
df_base = pd.read_csv(df_base_path)
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None) 

In [7]:
#4. Me quedo con un registro por día de la respuesta del LLM.
from news_daily import NewsDailyAggregator

agg = NewsDailyAggregator(
    date_col="fecha",
    percentiles=(0.1, 0.25, 0.5, 0.75, 0.9),
    list_like_cols=("empresas_mencionadas_list", "personas_mencionadas"),
    list_null_as_empty=True,
    coerce_listlike=True,
    numeric_impute_map={"horizonte_dias": 0},
    object_impute_map={"categoria_fuente": "desconocido"},
    object_topk=3, object_topk_includes=("categoria_fuente",),
    list_topk=10,  # top-10 personas/empresas daily rate features
)

agg.fit(df_llm)
df_llm_daily = agg.transform(df_llm)
features = agg.get_feature_names_out()
print(len(features), "aggregated features")

922 aggregated features


In [ ]:
#5. Joineo con el dataframe base.

In [ ]:
#6. Creación de lags.
df_lags = pd.DataFrame()

In [ ]:
#7. Feature selection.
from feature_sel import run_feature_selection

# Suppose df_daily has 'fecha', your aggregated features, and target 'y_t_plus_h'
X_selected, selector, report = run_feature_selection(
    df=df_lags, 
    target_col="y_t_plus_h",              # <- your aligned forecasting target (e.g., return at t+h)
    mode="boosting",                      # or "lstm"
    forbid_patterns=[r"(^|_)target(_|$)"],# optional: drop obvious proxies
    max_features=64,
    score_quantile=0.50,
    corr_threshold=0.95,
)

print("Kept", len(selector.get_support()), "features")
print("Dropped for high-missing:", len(report.dropped_high_missing))
print("Top 10 by combined score:",
      sorted(report.scores.combined.items(), key=lambda kv: kv[1], reverse=True)[:10])

In [ ]:
#8. Exporto dataset final.

In [ ]:
# Siguientes pasos.
#a. Feature selection.
#b. Entrenamiento.
#c. Predicción.
#d. Comparación modelo base vs modelo con openai.
#e. Conclusiones.